# AI vs Human Generated Image Classification
### CNN with Wavelet (DWT) · DCT · Gabor Feature Extraction

**Team:** Abhinav I · Fawas P · Jifsha Jabir · Niharika Ranjith · Sreelakshmi M M · Usmanul Faris

---
**Pipeline:**
1. Download dataset (Kaggle)
2. EDA — visualise AI vs Real images
3. Preprocessing — DWT noise · DCT artifacts · Gabor texture → 3-channel feature stack
4. Train CNN (20 epochs, Adam, Binary Cross-Entropy)
5. Evaluate & predict

## 1. Dataset Download

In [ ]:
import kagglehub

path = kagglehub.dataset_download('alessandrasala79/ai-vs-human-generated-dataset')
print('Dataset path:', path)

## 2. Load CSVs & Inspect

In [ ]:
import pandas as pd

train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')

print('Train shape:', train_df.shape)
print(train_df.head())
print('\nLabel distribution:')
print(train_df['label'].value_counts())

## 3. EDA — Visualise AI vs Real Images with Canny Edges

In [ ]:
import os, random, cv2
import numpy as np
import matplotlib.pyplot as plt

DATASET_PATH = 'data/train_data'

real_images = train_df[train_df['label'] == 0]['file_name'].tolist()
ai_images   = train_df[train_df['label'] == 1]['file_name'].tolist()

selected_ai   = random.sample(ai_images,   5)
selected_real = random.sample(real_images, 5)

fig, axes = plt.subplots(5, 4, figsize=(20, 25))

for i in range(5):
    for j, (label, img_list) in enumerate(zip(['AI Generated', 'Real'],
                                               [selected_ai, selected_real])):
        img_path = os.path.join(DATASET_PATH, img_list[i])
        img      = cv2.imread(img_path)
        if img is None: continue
        img_gray   = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        col_offset = j * 2
        axes[i, col_offset].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i, col_offset].set_title(f'{label} Image {i+1}')
        axes[i, col_offset].axis('off')
        edges = cv2.Canny(img_gray, 100, 200)
        axes[i, col_offset+1].imshow(edges, cmap='gray')
        axes[i, col_offset+1].set_title(f'{label} Canny Edges')
        axes[i, col_offset+1].axis('off')

plt.tight_layout()
plt.show()

## 4. Preprocessing — DWT + DCT + Gabor → Feature Stack

In [ ]:
import pywt
from tqdm.notebook import tqdm

OUTPUT_DIR = 'data/preprocessed_dataset'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def compute_wavelet_noise(img_gray):
    _, (LH, HL, HH) = pywt.dwt2(img_gray, 'haar')
    return np.abs(HH)

def compute_dct_artifacts(img_gray):
    dct = cv2.dct(np.float32(img_gray))
    return np.log(np.abs(dct) + 1)

def compute_gabor(img_gray):
    k = cv2.getGaborKernel((5,5), 1.0, 0, 10.0, 0.5, 0, ktype=cv2.CV_32F)
    return cv2.filter2D(img_gray, cv2.CV_8UC3, k)

def preprocess_and_save(image_list, dataset_path, output_dir, img_size=64):
    for img_name in tqdm(image_list, desc='Preprocessing'):
        img = cv2.imread(os.path.join(dataset_path, os.path.basename(img_name)),
                         cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        noise = cv2.resize(compute_wavelet_noise(img), (img_size, img_size))
        dct   = cv2.resize(compute_dct_artifacts(img), (img_size, img_size))
        gabor = cv2.resize(compute_gabor(img),          (img_size, img_size))
        for arr in [noise, dct, gabor]:
            cv2.normalize(arr, arr, 0, 255, cv2.NORM_MINMAX)
        stack = np.dstack([noise, dct, gabor]).astype(np.uint8)
        cv2.imwrite(os.path.join(output_dir, os.path.basename(img_name)), stack)

preprocess_and_save(train_df['file_name'].tolist(), DATASET_PATH, OUTPUT_DIR)
print('Done!')

## 5. Build & Train CNN

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.model_selection import train_test_split

# Load preprocessed images
labels_dict = dict(zip(train_df['file_name'].apply(os.path.basename), train_df['label']))
X, y = [], []

for img_name in tqdm(os.listdir(OUTPUT_DIR), desc='Loading'):
    if img_name not in labels_dict: continue
    img = cv2.imread(os.path.join(OUTPUT_DIR, img_name), cv2.IMREAD_UNCHANGED)
    if img is None: continue
    img = cv2.resize(img, (64, 64)) / 255.0
    X.append(img)
    y.append(labels_dict[img_name])

X = np.array(X, dtype=np.float32)
y = np.array(y)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2,
                                                    random_state=42, stratify=y)

model = Sequential([
    Conv2D(32,  (3,3), activation='relu', input_shape=(64,64,3)),
    MaxPooling2D((2,2)),
    Conv2D(64,  (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid'),
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(X_train, y_train, epochs=20, batch_size=32,
                    validation_data=(X_val, y_val))

os.makedirs('models', exist_ok=True)
model.save('models/ai_vs_real_classifier.h5')
print('Model saved!')

## 6. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'],     label='Train Acc')
ax1.plot(history.history['val_accuracy'], label='Val Acc')
ax1.set_title('Accuracy'); ax1.legend()

ax2.plot(history.history['loss'],     label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Loss'); ax2.legend()

plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=150)
plt.show()

## 7. Predict on a Single Image

In [ ]:
def preprocess_image(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None, None
    noise = cv2.resize(compute_wavelet_noise(img), (img.shape[1], img.shape[0]))
    dct   = cv2.resize(compute_dct_artifacts(img), (img.shape[1], img.shape[0]))
    gabor = cv2.resize(compute_gabor(img),          (img.shape[1], img.shape[0]))
    for arr in [noise, dct, gabor]:
        cv2.normalize(arr, arr, 0, 255, cv2.NORM_MINMAX)
    stack = np.dstack([noise, dct, gabor]).astype(np.uint8)
    stack = cv2.resize(stack, (64, 64)) / 255.0
    return stack, img

# Replace with your test image path
TEST_IMAGE = 'data/test_data_v2/your_test_image.jpg'

input_image, original_image = preprocess_image(TEST_IMAGE)
if input_image is not None:
    pred       = model.predict(np.expand_dims(input_image, 0), verbose=0)[0][0]
    label      = 'AI Generated' if pred > 0.5 else 'Real'
    confidence = pred if pred > 0.5 else 1 - pred
    print(f'Prediction: {label}  (Confidence: {confidence:.2f})')
    plt.figure(figsize=(5,5))
    plt.imshow(original_image, cmap='gray')
    plt.title(f'Prediction: {label}\nConfidence: {confidence:.2f}')
    plt.axis('off')
    plt.show()

## 8. Batch Predict → submission.csv

In [ ]:
TEST_DIR = 'data/test_data_v2'
results  = []

for img_name in tqdm(os.listdir(TEST_DIR), desc='Predicting'):
    if not img_name.lower().endswith(('.jpg','.jpeg','.png')): continue
    inp, _ = preprocess_image(os.path.join(TEST_DIR, img_name))
    if inp is None: continue
    pred  = model.predict(np.expand_dims(inp, 0), verbose=0)[0][0]
    label = 'AI Generated' if pred > 0.5 else 'Real'
    results.append({'image_name': img_name, 'predicted_label': label})

sub = pd.DataFrame(results)
sub.to_csv('outputs/submission.csv', index=False)
print(sub['predicted_label'].value_counts())